In [23]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("test")
    .getOrCreate()
)

In [24]:
# Option 1
import glob

green_paths = glob.glob("../data/pq/green/*/*")
yellow_paths = glob.glob("../data/pq/yellow/*/*")

df_green = spark.read.parquet(*green_paths)
df_yellow = spark.read.parquet(*yellow_paths)

print(df_green.count(), df_yellow.count())

8348567 124048218


In [25]:
# Option 2

df_green = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("../data/pq/green")
)

df_yellow = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("../data/pq/yellow")
)

print(df_green.count(), df_yellow.count())

8348567 124048218


In [26]:
df_green.take(5)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2019, 1, 29, 15, 36, 35), lpep_dropoff_datetime=datetime.datetime(2019, 1, 29, 15, 45, 15), store_and_fwd_flag='N', RatecodeID=1, PULocationID=40, DOLocationID=181, passenger_count=1, trip_distance=1.12, fare_amount=7.5, extra=0.0, mta_tax=0.5, tip_amount=1.66, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=9.96, payment_type=1, trip_type=1, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2019, 1, 12, 5, 8, 28), lpep_dropoff_datetime=datetime.datetime(2019, 1, 12, 5, 8, 33), store_and_fwd_flag='N', RatecodeID=1, PULocationID=74, DOLocationID=74, passenger_count=1, trip_distance=0.03, fare_amount=2.5, extra=0.5, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=3.8, payment_type=2, trip_type=1, congestion_surcharge=None),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2019, 1, 27, 0, 55, 22), lpep_dropoff_date

In [27]:
df_yellow.take(5)

[Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2019, 3, 23, 13, 15, 29), tpep_dropoff_datetime=datetime.datetime(2019, 3, 23, 13, 19, 57), passenger_count=1, trip_distance=1.0, RatecodeID=1, store_and_fwd_flag='N', PULocationID=237, DOLocationID=141, payment_type=1, fare_amount=5.5, extra=2.5, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=0.3, total_amount=10.3, congestion_surcharge=2.5),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2019, 3, 20, 14, 44, 26), tpep_dropoff_datetime=datetime.datetime(2019, 3, 20, 15, 3, 52), passenger_count=2, trip_distance=2.0, RatecodeID=1, store_and_fwd_flag='N', PULocationID=162, DOLocationID=50, payment_type=1, fare_amount=13.5, extra=0.0, mta_tax=0.5, tip_amount=4.2, tolls_amount=0.0, improvement_surcharge=0.3, total_amount=21.0, congestion_surcharge=2.5),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2019, 3, 12, 22, 55, 6), tpep_dropoff_datetime=datetime.datetime(2019, 3, 12, 23, 11, 14), passen

In [28]:
set(df_green.columns) & set(df_yellow.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}

In [29]:
common_columns = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_columns.append(col)

In [31]:
from pyspark.sql import functions as F 

df_green_sel = (
    df_green.select(
        *common_columns,
        F.lit("green").alias("service_type"),
        F.col("lpep_pickup_datetime").alias("pickup_datetime"),
        F.col("lpep_dropoff_datetime").alias("dropoff_datetime")
    )
)
df_yellow_sel = (
    df_yellow.select(
        *common_columns,
        F.lit("yellow").alias("service_type"),
        F.col("tpep_pickup_datetime").alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
    )
)

df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [32]:
df_trips_data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- service_type: string (nullable = false)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)



In [33]:
df_trips_data.groupBy("service_type").count().show()

+------------+---------+
|service_type|    count|
+------------+---------+
|       green|  8348567|
|      yellow|124048218|
+------------+---------+



In [34]:
# Tell spark that this dataframe is a table
# This method is deprecated
# df_trips_data.registerTempTable("trips_data")

# Use this method instead
df_trips_data.createOrReplaceTempView("trips_data")

In [35]:
spark.sql("""
SELECT
    service_type,
    count(*)
FROM
    trips_data
GROUP BY
    service_type
""").show()

+------------+---------+
|service_type| count(1)|
+------------+---------+
|       green|  8348567|
|      yellow|124048218|
+------------+---------+



In [37]:
df_result = spark.sql("""
SELECT
    -- Revenue grouping
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month,
    service_type,

    -- Revenue calculation
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_monthly_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [38]:
df_result.show()

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+-------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_monthly_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+-------------------------+-------------------------+
|         200|2019-

In [42]:
spark.createDataFrame(df_result.tail(10)).show()

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+-------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_monthly_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+-------------------------+-------------------------+
|           7|2020-

In [43]:
df_result.write.parquet("../data/report/revenue/", mode="overwrite")

In [44]:
df_result.coalesce(1).write.parquet("../data/report/revenue/", mode="overwrite")